<a href="https://colab.research.google.com/github/n-schiozer/met_comp_economistas_unesp/blob/main/aula6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Métodos computacionais para economistas
## Aula 6 — Pandas II: importação de dados e APIs

**Pergunta da aula:** como obter dados econômicos publicados na internet, compreender seu formato e transformá-los em um `DataFrame` utilizável?

**Objetivos**

- Diferenciar **Excel**, **CSV** e **JSON** e identificar as vantagens e limitações de cada formato.
- Ler um arquivo CSV com `pd.read_csv()`, observando **separador de colunas, separador decimal e valores ausentes**.
- Entender o que é uma **API** e como funciona, de forma introdutória, uma requisição HTTP.
- Consultar o **Sistema Gerenciador de Séries Temporais (SGS)** do Banco Central do Brasil.
- Converter datas e valores para os tipos apropriados e salvar uma base documentada.
- Interpretar corretamente a periodicidade e as unidades de uma série de inflação.


**Estudo complementar para esta aula:** [Pandas — entrada e saída de dados](https://pandas.pydata.org/docs/user_guide/io.html), [Requests — primeiros passos](https://requests.readthedocs.io/en/latest/user/quickstart/) e [BCB — consulta à série SGS 433](https://www3.bcb.gov.br/sgspub/consultarvalores/consultarValoresSeries.do?method=consultarSeries&series=433).

## 1. Da planilha local aos dados disponíveis na internet

Na Aula 5, abrimos o arquivo `dados_brasil_2012_2025.xlsx` com `pd.read_excel()` e trabalhamos com suas colunas. Hoje vamos aprender a obter **outros formatos de dados**.

| Formato | Como os dados costumam aparecer | Leitura em Python |
|---|---|---|
| Excel (`.xlsx`) | Planilha com abas e células | `pd.read_excel()` |
| CSV (`.csv`) | Arquivo de **texto**: uma linha por observação, colunas separadas por um caractere | `pd.read_csv()` |
| JSON (`.json`) | Listas e dicionários que representam informações estruturadas | `pd.DataFrame()`, `response.json()` |

**O que não muda:** depois da importação, podemos usar os métodos de Pandas conhecidos da aula anterior.

**Uma distinção:** *formato do arquivo* (CSV, JSON, Excel) não é a mesma coisa que *fonte dos dados* (IBGE, Banco Central etc.).

**Leitura:** [Pandas — guia de importação/exportação](https://pandas.pydata.org/docs/user_guide/io.html).

### 1.1. CSV não é Excel: separadores importam

Considere o arquivo fictício abaixo, no qual `;` separa **colunas** e `,` separa as **casas decimais**. A abreviação `NA` significa informação ausente.

```text
ano;taxa_ficticia_pct
2022;0,40
2023;0,75
2024;NA
2025;0,55
```

Os valores são **inteiramente fictícios**, criados apenas para explicar o formato. **Não são dados oficiais de inflação.**

`pd.read_csv()` tem parâmetros que ajudam a interpretar corretamente esse texto:

- `sep=";"`: separador entre colunas;
- `decimal=","`: separador decimal;
- `na_values=["NA"]`: marca `NA` como valor ausente.

**Leitura:** [Documentação de `pandas.read_csv()`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html).

In [32]:
import pandas as pd
from io import StringIO

# Dados inventados para demonstrar a leitura de CSV.
csv_exemplo = """ano;taxa_ficticia_pct
2022;0,40
2023;0,75
2024;NA
2025;0,55
"""

# StringIO permite tratar um texto como se fosse um arquivo aberto.
dados_csv = pd.read_csv(
    StringIO(csv_exemplo),
    sep=";",
    decimal=",",
    na_values=["NA"]
)

dados_csv

,ano,taxa_ficticia_pct
0,2022,0.40
1,2023,0.75
2,2024,NaN
3,2025,0.55


In [33]:
print("Dimensões:", dados_csv.shape)
print("Tipos das colunas:")
print(dados_csv.dtypes)
print("Quantidade de valores ausentes por coluna:")
print(dados_csv.isna().sum())

Dimensões: (4, 2)
Tipos das colunas:
ano                    int64
taxa_ficticia_pct    float64
dtype: object
Quantidade de valores ausentes por coluna:
ano                  0
taxa_ficticia_pct    1
dtype: int64


### Exercício 1 — decifrar um arquivo CSV

1. A coluna `taxa_ficticia_pct` foi reconhecida como numérica? Por quê?
2. Troque `decimal=","` por `decimal="."` e observe o que muda. **Depois volte à configuração correta.**
3. Quantas observações existem? Quantas informações estão ausentes?
4. Por que não devermos calcular a média sem antes inspecionar o tipo e os valores ausentes?

Use as células abaixo para registrar suas respostas e testar comandos.

**Leitura de apoio:** [Pandas — dados ausentes](https://pandas.pydata.org/docs/user_guide/missing_data.html).

In [34]:
# Exercício 1 — seus comandos
# 1) Exiba apenas a coluna "taxa_ficticia_pct".
# 2) Conte os valores não ausentes.
# 3) Calcule a média dos valores disponíveis.
# Escreva seu código aqui.

## 2. O que é uma API?

Uma **API** (*Application Programming Interface*) permite que um programa solicite informações a outro sistema, sem precisar abrir manualmente uma página e copiar os resultados.

No nosso caso:

1. O Python envia uma **requisição** para um endereço do Banco Central.
2. O servidor responde com informações de uma série econômica.
3. A resposta vem em **JSON**: muitas vezes, uma **lista de dicionários** (conteúdo já conhecido das Aulas 2 e 3).
4. Transformamos a resposta em um `DataFrame` para trabalhar com Pandas.

Um *endpoint* é o endereço de um serviço específico da API. A palavra **GET** designa a operação usada para pedir/consultar dados. Uma resposta bem-sucedida costuma usar o código HTTP **200**, mas devemos verificar erros antes de interpretar os dados.

**Leitura:** [Requests — requisições e respostas](https://requests.readthedocs.io/en/latest/user/quickstart/) e [Documentação Python — JSON](https://docs.python.org/3/library/json.html).

### 2.1. JSON: conexão com listas e dicionários

Primeiro, vamos simular o formato de uma resposta. **Os valores abaixo são fictícios**: servem apenas para entender como listas e dicionários são representados.

No exemplo, a variável `resposta_exemplo` é uma **lista**. Cada elemento da lista é um **dicionário** com as chaves `data` e `valor`.

In [35]:
resposta_exemplo = [
    {"data": "01/01/2024", "valor": "0.40"},
    {"data": "01/02/2024", "valor": "0.75"}
]

print("Tipo do objeto:", type(resposta_exemplo))
print("Primeiro elemento:", resposta_exemplo[0])
print("Primeiro valor:", resposta_exemplo[0]["valor"])

exemplo_dataframe = pd.DataFrame(resposta_exemplo)
exemplo_dataframe

Tipo do objeto: <class 'list'>
Primeiro elemento: {'data': '01/01/2024', 'valor': '0.40'}
Primeiro valor: 0.40


,data,valor
0,01/01/2024,0.40
1,01/02/2024,0.75


**Observe:** embora pareça um número, `"0.40"` está entre aspas e, portanto, é uma **string** (*texto*). Ao receber dados externos, devemos conferir os tipos antes de fazer cálculos.

**Perguntas rápidas:** o que muda entre `resposta_exemplo[0]` e `resposta_exemplo[0]["valor"]`? E como seria necessário organizar a resposta se uma série apresentasse três variáveis por mês?

**Leitura:** [Think Python — dicionários](https://allendowney.github.io/ThinkPython/chap11.html) e [Pandas — estruturas de dados](https://pandas.pydata.org/docs/user_guide/dsintro.html).

## 3. Aplicação: consultar o IPCA mensal no Banco Central

O **IPCA** é produzido pelo IBGE; o **Banco Central republica séries econômicas** pelo Sistema Gerenciador de Séries Temporais (SGS). Vamos acessar a série **SGS 433 — IPCA, variação percentual mensal**, para os anos de **2024 e 2025**.

- **Indicador:** IPCA, variação mensal.
- **Unidade:** percentual (%) no mês. Por exemplo, o valor `0.50` na API significa **0,50% no mês**, e não 50%.
- **Periodicidade:** mensal.
- **Fonte primária da estatística:** IBGE.
- **Distribuidor da API utilizada:** BCB/SGS.
- **Série:** código `433`.
- **Intervalo da consulta:** 01/01/2024 a 31/12/2025 (fixo para permitir reprodução da atividade).

**Por que não usar uma taxa anual?** A planilha da Aula 5 trazia o IPCA **acumulado no ano**; agora veremos as **variações mensais**, que são medidas diferentes.

**Fontes oficiais para conferir conceito e dados:** [BCB/SGS — série 433](https://www3.bcb.gov.br/sgspub/consultarvalores/consultarValoresSeries.do?method=consultarSeries&series=433) e [IBGE — IPCA](https://www.ibge.gov.br/estatisticas/economicas/precos-e-custos/9256-indice-nacional-de-precosconsumidor-amplo.html).

### 3.1. Como escrever a requisição

Usaremos a biblioteca `requests`:

```python
requests.get(endereco, params=parametros, timeout=15)
```

- `endereco`: onde está o serviço;
- `params`: dicionário com as opções da consulta;
- `formato=json`: tipo de resposta desejado;
- `dataInicial` e `dataFinal`: início e fim do intervalo, no padrão **dia/mês/ano**;
- `timeout=15`: evita que o programa espere indefinidamente;
- `.raise_for_status()`: verifica erros HTTP;
- `.json()`: transforma a resposta JSON em objetos Python.

**Internet:** a consulta ao Banco Central depende de conexão. Caso o serviço esteja temporariamente indisponível, a célula prepara **dados simulados e claramente identificados como tal**, exclusivamente para praticar importação e manipulação. **Não use os dados simulados para conclusões econômicas.**

**Leitura:** [Requests — parâmetros de URL, JSON, erros e timeout](https://requests.readthedocs.io/en/latest/user/quickstart/).

In [36]:
import requests

url = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados"
parametros = {
    "formato": "json",
    "dataInicial": "01/01/2024",
    "dataFinal": "31/12/2025"
}

try:
    resposta = requests.get(url, params=parametros, timeout=15)
    resposta.raise_for_status()
    registros = resposta.json()
    if not isinstance(registros, list) or not registros:
        raise ValueError("A API retornou uma estrutura vazia ou inesperada.")
    origem_dados = "OFICIAL — Banco Central/SGS, série 433 (fonte primária IBGE)"
    print("Consulta bem-sucedida!")
    print("Código HTTP:", resposta.status_code)
    print("URL consultada:", resposta.url)

except (requests.exceptions.RequestException, ValueError) as erro:
    # Reserva pedagógica: 24 observações INVENTADAS, não dados do IBGE/BCB.
    # O restante do notebook pode funcionar sem internet.
    datas_simuladas = pd.date_range("2024-01-01", periods=24, freq="MS")
    valores_simulados = [
        0.40, 0.55, 0.25, 0.30, 0.45, 0.20, 0.35, 0.10, 0.60, 0.50, 0.40, 0.65,
        0.30, 0.45, 0.35, 0.25, 0.50, 0.15, 0.20, 0.40, 0.55, 0.35, 0.30, 0.50
    ]
    registros = [
        {"data": data.strftime("%d/%m/%Y"), "valor": str(valor)}
        for data, valor in zip(datas_simuladas, valores_simulados)
    ]
    origem_dados = "SIMULADOS — valores fictícios apenas para prática"
    print("Não foi possível consultar o serviço:", erro)
    print("ATENÇÃO: esta execução está utilizando DADOS SIMULADOS.")

print("Origem:", origem_dados)
print("Registros recebidos:", len(registros))
print("Duas primeiras observações:", registros[:2])

Consulta bem-sucedida!
Código HTTP: 200
URL consultada: https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json&dataInicial=01%2F01%2F2024&dataFinal=31%2F12%2F2025
Origem: OFICIAL — Banco Central/SGS, série 433 (fonte primária IBGE)
Registros recebidos: 24
Duas primeiras observações: [{'data': '01/01/2024', 'valor': '0.42'}, {'data': '01/02/2024', 'valor': '0.83'}]


**Antes de continuar:** confirme que a mensagem `Origem:` informa se os dados são **oficiais** ou **simulados**. Se aparecer `SIMULADOS`, vale testar a consulta novamente com internet antes de interpretar valores econômicos.

**Pergunta:** em qual etapa nós obtivemos objetos Python? Em qual etapa transformaremos esses objetos em uma tabela?

### 3.2. Da resposta JSON ao `DataFrame`

A API apresenta os campos `data` e `valor`. Precisamos garantir que as datas sejam reconhecidas como datas e que as taxas de inflação sejam números.

- `pd.DataFrame(registros)`: cria uma tabela a partir da lista de dicionários.
- `pd.to_datetime(..., format="%d/%m/%Y")`: interpreta datas como `dia/mês/ano`.
- `pd.to_numeric(...)`: converte números representados como texto em valores numéricos.
- `.rename(columns={...})`: dá um nome explícito à variável econômica.

**Cuidado:** a conversão de tipos não verifica, por si só, se a série tem a periodicidade correta nem se seus valores são economicamente plausíveis.

**Leituras:** [Pandas — `to_datetime`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html), [Pandas — `to_numeric`](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html).

In [37]:
ipca = pd.DataFrame(registros)
print("Antes das conversões:")
print(ipca.dtypes)
ipca.head(3)

Antes das conversões:
data     object
valor    object
dtype: object


,data,valor
0,01/01/2024,0.42
1,01/02/2024,0.83
2,01/03/2024,0.16


In [38]:
ipca["data"] = pd.to_datetime(ipca["data"], format="%d/%m/%Y")
ipca["valor"] = pd.to_numeric(ipca["valor"], errors="coerce")

ipca = ipca.rename(columns={"valor": "ipca_mensal_pct"})
ipca = ipca.sort_values("data").reset_index(drop=True)

print("Origem:", origem_dados)
print("Depois das conversões:")
print(ipca.dtypes)
ipca.head(3)

Origem: OFICIAL — Banco Central/SGS, série 433 (fonte primária IBGE)
Depois das conversões:
data               datetime64[ns]
ipca_mensal_pct           float64
dtype: object


,data,ipca_mensal_pct
0,2024-01-01,0.42
1,2024-02-01,0.83
2,2024-03-01,0.16


### 3.3. O que conferir antes de analisar?

Uma importação executada sem erro **não garante qualidade dos dados**. Sempre verifique:

1. **Período** — a base cobre os meses pretendidos?
2. **Frequência** — há uma observação por mês?
3. **Valores ausentes** — há lacunas?
4. **Duplicatas** — algum mês aparece mais de uma vez?
5. **Unidade e definição** — porcentagem mensal é diferente de taxa acumulada no ano.

**Leitura:** [Pandas — inspeção e valores ausentes](https://pandas.pydata.org/docs/user_guide/missing_data.html).

In [39]:
print("Fonte/qualidade da série:", origem_dados)
print("Observações:", len(ipca))
print("Primeira data:", ipca["data"].min())
print("Última data:", ipca["data"].max())
print("Valores ausentes:")
print(ipca.isna().sum())
print("Meses duplicados:", ipca["data"].duplicated().sum())

# 2024 e 2025 completos têm 12 + 12 = 24 observações mensais.
print("Há 24 observações?", len(ipca) == 24)

Fonte/qualidade da série: OFICIAL — Banco Central/SGS, série 433 (fonte primária IBGE)
Observações: 24
Primeira data: 2024-01-01 00:00:00
Última data: 2025-12-01 00:00:00
Valores ausentes:
data               0
ipca_mensal_pct    0
dtype: int64
Meses duplicados: 0
Há 24 observações? True


### Exercício 2 — investigar a série

Usando a base `ipca`, escreva comandos para:

1. Mostrar as **cinco últimas** observações.
2. Selecionar apenas as colunas `data` e `ipca_mensal_pct`.
3. Descobrir a **maior** e a **menor** taxa mensal da série (ainda sem identificar o mês).
4. Identificar quantos meses tiveram taxa mensal **negativa**, se houver.
5. Explicar por que um IPCA mensal negativo **não implica**, necessariamente, inflação anual negativa.

**Dica:** reutilize `.tail()`, seleção de colunas, `.max()`, `.min()` e filtros da Aula 5.

**Leitura:** [Pandas — seleção e filtragem de dados](https://pandas.pydata.org/docs/user_guide/indexing.html).

In [40]:
# Exercício 2 — completar usando a base ipca
# 1) Últimas cinco linhas:

# 2) Apenas as duas colunas de interesse:

# 3) Maior e menor taxa mensal:

# 4) Quantidade de meses com taxa menor do que zero:

## 4. Aplicação econômica: inflação mensal e inflação acumulada

A série SGS 433 traz **taxas mensais** de variação dos preços. Para calcular a inflação acumulada em um conjunto de meses consecutivos, combinamos os fatores de variação:

$$\pi_{acum} = \left[\prod_{m=1}^{M}\left(1+\frac{\pi_m}{100}\right)-1\right]\times100,$$

onde $\pi_m$ é a inflação mensal expressa em porcentagem (por exemplo, `0.50` para 0,50%).

**Por que não somar?** Porque o preço de um mês varia sobre o nível de preços já alterado no mês anterior. Somar taxas mensais dá apenas uma aproximação, não a variação acumulada exata.

Em Python, podemos criar uma função e aplicá-la à coluna escolhida, retomando a **Aula 4**.

**Fonte metodológica:** [IBGE — cálculo do IPCA acumulado e correção monetária](https://www.ibge.gov.br/estatisticas/economicas/precos-e-custos/9256-indice-nacional-de-precosconsumidor-amplo.html?edicao=20932&t=series-historicas).

In [41]:
def calcular_inflacao_acumulada(taxas_mensais):
    """Recebe taxas mensais em %; devolve a inflação acumulada em %."""
    fatores = 1 + taxas_mensais / 100
    return (fatores.prod() - 1) * 100

# Exemplo isolado para compreender a fórmula (valores fictícios):
taxas_exemplo = pd.Series([1.0, 1.0])
print("Duas altas de 1% se acumulam em:", calcular_inflacao_acumulada(taxas_exemplo), "%")

Duas altas de 1% se acumulam em: 2.0100000000000007 %


### Exercício 3 — usar dados reais (ou simulação, caso esteja sem internet)

1. Selecione os meses de **2025**.
2. Calcule a inflação acumulada em 2025 com a função `calcular_inflacao_acumulada`.
3. Calcule a soma simples das taxas mensais de 2025 e compare as duas medidas.
4. Explique por que os resultados não precisam ser exatamente iguais.
5. Se estiver utilizando dados **oficiais**, compare o resultado com o IPCA acumulado em 2025 divulgado pelo IBGE ou com a coluna correspondente da planilha da Aula 5. Se estiver utilizando dados **simulados**, **não** faça essa comparação: os valores são inventados.

**Dica:** para extrair o ano de uma coluna de datas use `ipca["data"].dt.year`. Isso é uma aplicação de atributos de datas no Pandas; veremos manipulações mais aprofundadas na próxima aula.

**Leituras:** [Pandas — componentes de datas](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.html) e [IBGE — IPCA](https://www.ibge.gov.br/estatisticas/economicas/precos-e-custos/9256-indice-nacional-de-precosconsumidor-amplo.html).

In [42]:
# Exercício 3 — complete as lacunas
# 1) Selecione 2025, usando a condição ipca["data"].dt.year == 2025.
#    Salve a seleção em uma variável chamada ipca_2025.

# 2) Passe a coluna ipca_2025["ipca_mensal_pct"] à função da aula.

# 3) Compare o resultado com a soma simples das taxas.

# 4) Imprima a variável origem_dados para distinguir dado real de simulado.

## 5. Guardar o resultado e documentar a origem

Depois de obter e preparar a base, podemos exportá-la. Na Aula 5, usamos `to_excel()`. Para arquivos de texto, usamos `to_csv()`.

**Boa prática:** junto da base, registrar a **fonte**, o **código da série**, a **unidade**, o **período da consulta** e a **data de acesso**. Isso ajuda a reproduzir a pesquisa e descobrir se uma base foi alterada.

Neste notebook, o nome do arquivo também distingue dados **oficiais** de dados **simulados**.

**Leitura:** [Pandas — `DataFrame.to_csv()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html).

In [43]:
nome_arquivo = (
    "ipca_bcb_2024_2025.csv"
    if origem_dados.startswith("OFICIAL")
    else "exemplo_simulado_ipca_2024_2025_NAO_OFICIAL.csv"
)

ipca.to_csv(
    nome_arquivo,
    sep=";",
    decimal=",",
    date_format="%d/%m/%Y",
    index=False
)

print("Arquivo gerado:", nome_arquivo)
print("Origem:", origem_dados)

# Confirme que é possível ler o arquivo recém-gravado.
conferencia = pd.read_csv(nome_arquivo, sep=";", decimal=",")
conferencia.head(3)

Arquivo gerado: ipca_bcb_2024_2025.csv
Origem: OFICIAL — Banco Central/SGS, série 433 (fonte primária IBGE)


,data,ipca_mensal_pct
0,01/01/2024,0.42
1,01/02/2024,0.83
2,01/03/2024,0.16


### Exercício 4 — documentação e reprodutibilidade

Imagine que outro estudante receba apenas o arquivo CSV exportado, sem acesso ao notebook.

1. Que **informações** ele deveria receber sobre a base para utilizá-la adequadamente?
2. Quais problemas podem ocorrer ao ler o CSV usando o separador `,` em vez de `;`?
3. Se rodarmos a mesma consulta em outra data, a API necessariamente devolverá os mesmos dados? Por quê?
4. Altere apenas as datas da consulta à API para buscar **2023–2024**, execute novamente **a partir da seção 3.1** e explique que cuidados de documentação precisam ser atualizados.


**Leitura complementar:** [Banco Central — Portal de Dados Abertos](https://dadosabertos.bcb.gov.br/) e [Pandas — CSV](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html).

In [44]:
# Exercício 4 — espaço para experimentos
# Teste a leitura do arquivo que você acabou de exportar.
# Depois, explique por escrito as informações que deveriam acompanhá-lo.

## 6. Síntese: o caminho percorrido

**Do dado bruto à base pronta para análise:**

`CSV / requisição à API` → `texto / JSON` → `DataFrame` → `conversão de tipos` → `verificação` → `análise` → `exportação`.

**Perguntas para discutir:**

- Ler uma planilha Excel e consultar uma API são maneiras diferentes de obter dados: **o que ambas têm em comum**?
- Por que é perigoso confundir **IPCA mensal**, **IPCA acumulado no ano** e **IPCA em 12 meses**?
- O fato de o código executar sem erros significa que a interpretação econômica está correta?

**Próxima aula:** preparação de bases no Pandas: filtros mais elaborados, valores ausentes, criação de variáveis e agrupamentos.

## Referências para estudar além do código

### Programação e Pandas

1. DOWNEY, Allen B. *Think Python*, 3ª ed. (livro aberto). [Capítulo 11 — Dicionários](https://allendowney.github.io/ThinkPython/chap11.html). Retomar listas e dicionários usados na estrutura de uma resposta JSON.
2. PANDAS DEVELOPMENT TEAM. *Pandas User Guide*. [Importação e exportação](https://pandas.pydata.org/docs/user_guide/io.html). Visão geral de arquivos CSV, Excel e JSON.
3. PANDAS DEVELOPMENT TEAM. *API Reference*. [`read_csv()`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html), [`to_datetime()`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html), [`to_numeric()`](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html), [`to_csv()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html). Consultar quando surgir dúvida sobre argumentos.
4. PYTHON SOFTWARE FOUNDATION. *Requests: Quickstart*. [Requisições HTTP, parâmetros, JSON e tratamento de erros](https://requests.readthedocs.io/en/latest/user/quickstart/). Para aprofundar o funcionamento das APIs.
5. PYTHON SOFTWARE FOUNDATION. *Python 3 — Biblioteca padrão*. [Módulo `json`](https://docs.python.org/3/library/json.html). Entender a representação e leitura de JSON.

### Fontes econômicas e metodologia

6. BANCO CENTRAL DO BRASIL. *Sistema Gerenciador de Séries Temporais (SGS)*. [Série 433 — IPCA mensal](https://www3.bcb.gov.br/sgspub/consultarvalores/consultarValoresSeries.do?method=consultarSeries&series=433). Identificação e histórico da série utilizada; [Portal de Dados Abertos](https://dadosabertos.bcb.gov.br/).
7. IBGE. *Índice Nacional de Preços ao Consumidor Amplo — IPCA*. [Conceito, metodologia, séries históricas e cálculo acumulado](https://www.ibge.gov.br/estatisticas/economicas/precos-e-custos/9256-indice-nacional-de-precosconsumidor-amplo.html). Fonte primária da estatística utilizada.

**Para aprofundar:** leia primeiro o [guia de CSV do Pandas](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) e o [Quickstart do Requests](https://requests.readthedocs.io/en/latest/user/quickstart/), depois consulte o [SGS 433](https://www3.bcb.gov.br/sgspub/consultarvalores/consultarValoresSeries.do?method=consultarSeries&series=433) para verificar unidade, periodicidade e origem da série. A execução do código não substitui a leitura da documentação.

In [45]:
# Gabarito do Exercício 1
print("Coluna numérica?", pd.api.types.is_numeric_dtype(dados_csv["taxa_ficticia_pct"]))
print("Tipo correto:", dados_csv["taxa_ficticia_pct"].dtype)

# Teste deliberadamente incorreto: usar ponto decimal em um CSV com vírgula.
leitura_decimal_incorreto = pd.read_csv(
    StringIO(csv_exemplo), sep=";", decimal=".", na_values=["NA"]
)
print("Tipo com decimal incorreto:", leitura_decimal_incorreto["taxa_ficticia_pct"].dtype)

print("Total de observações:", len(dados_csv))
print("Valores ausentes:", dados_csv["taxa_ficticia_pct"].isna().sum())
print("Valores não ausentes:", dados_csv["taxa_ficticia_pct"].count())
print("Média das taxas disponíveis:", dados_csv["taxa_ficticia_pct"].mean())

# Seleção da coluna solicitada no exercício.
print("\nColuna de taxas:")
print(dados_csv["taxa_ficticia_pct"])

Coluna numérica? True
Tipo correto: float64
Tipo com decimal incorreto: object
Total de observações: 4
Valores ausentes: 1
Valores não ausentes: 3
Média das taxas disponíveis: 0.5666666666666667

Coluna de taxas:
0    0.40
1    0.75
2     NaN
3    0.55
Name: taxa_ficticia_pct, dtype: float64


In [46]:
# Gabarito do Exercício 2
print("Origem dos dados:", origem_dados)

print("\n1) Cinco últimas observações:")
print(ipca.tail(5))

print("\n2) Apenas data e IPCA mensal:")
print(ipca[["data", "ipca_mensal_pct"]])

print("\n3) Maior taxa mensal:", ipca["ipca_mensal_pct"].max(), "%")
print("Menor taxa mensal:", ipca["ipca_mensal_pct"].min(), "%")

meses_negativos = (ipca["ipca_mensal_pct"] < 0).sum()
print("\n4) Meses com IPCA mensal negativo:", meses_negativos)

if not origem_dados.startswith("OFICIAL"):
    print("ATENÇÃO: resultados referentes a DADOS SIMULADOS, sem interpretação histórica.")

Origem dos dados: OFICIAL — Banco Central/SGS, série 433 (fonte primária IBGE)

1) Cinco últimas observações:
         data  ipca_mensal_pct
19 2025-08-01            -0.11
20 2025-09-01             0.48
21 2025-10-01             0.09
22 2025-11-01             0.18
23 2025-12-01             0.33

2) Apenas data e IPCA mensal:
         data  ipca_mensal_pct
0  2024-01-01             0.42
1  2024-02-01             0.83
2  2024-03-01             0.16
3  2024-04-01             0.38
4  2024-05-01             0.46
5  2024-06-01             0.21
6  2024-07-01             0.38
7  2024-08-01            -0.02
8  2024-09-01             0.44
9  2024-10-01             0.56
10 2024-11-01             0.39
11 2024-12-01             0.52
12 2025-01-01             0.16
13 2025-02-01             1.31
14 2025-03-01             0.56
15 2025-04-01             0.43
16 2025-05-01             0.26
17 2025-06-01             0.24
18 2025-07-01             0.26
19 2025-08-01            -0.11
20 2025-09-01         

In [47]:
# Gabarito do Exercício 3
ipca_2025 = ipca.loc[ipca["data"].dt.year == 2025].copy()
print("Origem:", origem_dados)
print("Meses de 2025 disponíveis:", len(ipca_2025))

if len(ipca_2025) != 12 or ipca_2025["ipca_mensal_pct"].isna().any():
    print("Base incompleta ou com valores ausentes: verificar antes de calcular a taxa anual.")
else:
    inflacao_acumulada_2025 = calcular_inflacao_acumulada(
        ipca_2025["ipca_mensal_pct"]
    )
    soma_simples_2025 = ipca_2025["ipca_mensal_pct"].sum()
    diferenca = inflacao_acumulada_2025 - soma_simples_2025

    print(f"Inflação acumulada (composição): {inflacao_acumulada_2025:.4f}%")
    print(f"Soma simples das taxas:          {soma_simples_2025:.4f}%")
    print(f"Diferença (pontos percentuais): {diferenca:.4f}")

    if origem_dados.startswith("OFICIAL"):
        print("Dados OFICIAIS: compare o acumulado com o IPCA anual publicado pelo IBGE.")
    else:
        print("DADOS SIMULADOS: NÃO comparar com estatísticas históricas do IBGE.")

# Exemplo isolado com números fictícios para verificar a diferença:
print("Duas variações de 1%:", calcular_inflacao_acumulada(pd.Series([1.0, 1.0])), "%")

Origem: OFICIAL — Banco Central/SGS, série 433 (fonte primária IBGE)
Meses de 2025 disponíveis: 12
Inflação acumulada (composição): 4.2644%
Soma simples das taxas:          4.1900%
Diferença (pontos percentuais): 0.0744
Dados OFICIAIS: compare o acumulado com o IPCA anual publicado pelo IBGE.
Duas variações de 1%: 2.0100000000000007 %


In [48]:
# Gabarito do Exercício 4 — conferir a leitura do CSV já exportado
arquivo_conferido = pd.read_csv(nome_arquivo, sep=";", decimal=",")
arquivo_conferido["data"] = pd.to_datetime(
    arquivo_conferido["data"], format="%d/%m/%Y"
)

print("Arquivo lido:", nome_arquivo)
print("Origem:", origem_dados)
print("Dimensões:", arquivo_conferido.shape)
print("Tipos das colunas:")
print(arquivo_conferido.dtypes)
print("Valores ausentes:")
print(arquivo_conferido.isna().sum())
print("Primeiras linhas:")
print(arquivo_conferido.head())

Arquivo lido: ipca_bcb_2024_2025.csv
Origem: OFICIAL — Banco Central/SGS, série 433 (fonte primária IBGE)
Dimensões: (24, 2)
Tipos das colunas:
data               datetime64[ns]
ipca_mensal_pct           float64
dtype: object
Valores ausentes:
data               0
ipca_mensal_pct    0
dtype: int64
Primeiras linhas:
        data  ipca_mensal_pct
0 2024-01-01             0.42
1 2024-02-01             0.83
2 2024-03-01             0.16
3 2024-04-01             0.38
4 2024-05-01             0.46


In [49]:
# Gabarito do Exercício 4 (item 4) — consulta independente a 2023–2024
# Esta consulta requer acesso ao serviço do BCB.
parametros_2023_2024 = {
    "formato": "json",
    "dataInicial": "01/01/2023",
    "dataFinal": "31/12/2024"
}

try:
    resposta_2023_2024 = requests.get(
        url, params=parametros_2023_2024, timeout=15
    )
    resposta_2023_2024.raise_for_status()
    registros_2023_2024 = resposta_2023_2024.json()

    if not isinstance(registros_2023_2024, list) or not registros_2023_2024:
        raise ValueError("A API não retornou observações válidas.")

    ipca_2023_2024 = pd.DataFrame(registros_2023_2024)
    ipca_2023_2024["data"] = pd.to_datetime(
        ipca_2023_2024["data"], format="%d/%m/%Y"
    )
    ipca_2023_2024["ipca_mensal_pct"] = pd.to_numeric(
        ipca_2023_2024["valor"], errors="coerce"
    )
    ipca_2023_2024 = (
        ipca_2023_2024[["data", "ipca_mensal_pct"]]
        .sort_values("data")
        .reset_index(drop=True)
    )

    nome_arquivo_2023_2024 = "ipca_bcb_2023_2024.csv"
    ipca_2023_2024.to_csv(
        nome_arquivo_2023_2024,
        sep=";", decimal=",", date_format="%d/%m/%Y", index=False
    )

    print("Origem: OFICIAL — BCB/SGS, série 433 (produtor: IBGE)")
    print("Período solicitado: 01/01/2023 a 31/12/2024")
    print("Data de acesso:", pd.Timestamp.now().strftime("%d/%m/%Y"))
    print("Arquivo:", nome_arquivo_2023_2024)
    print("Observações recebidas:", len(ipca_2023_2024))
    print(ipca_2023_2024.head(3))

except (requests.exceptions.RequestException, ValueError, KeyError) as erro:
    print("A consulta oficial de 2023–2024 não pôde ser concluída:", erro)
    print("Sem dados oficiais, não há série de 2023–2024 a relatar.")

Origem: OFICIAL — BCB/SGS, série 433 (produtor: IBGE)
Período solicitado: 01/01/2023 a 31/12/2024
Data de acesso: 26/10/2026
Arquivo: ipca_bcb_2023_2024.csv
Observações recebidas: 24
        data  ipca_mensal_pct
0 2023-01-01             0.53
1 2023-02-01             0.84
2 2023-03-01             0.71
